# Part 3 — Feature Engineering

Build the feature matrix for the full 2023–2025 dataset.  
Key additions over Part 2: multi-year lag features, seasonal dummies, and route/carrier history computed on the full corpus.

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import config

config.assert_data_exists()

## 1. Load Raw Data

In [2]:
df = config.load_bts_flights()
print(df.shape)

(20928599, 23)


## 2. Parse Datetime Fields

In [3]:
# YEAR, MONTH, DAY_OF_WEEK are already columns in the raw data
# Parse FL_DATE only for cyclical/ordinal features
df["fl_date"] = pd.to_datetime(df["FL_DATE"])
df["day_of_year"]  = df["fl_date"].dt.dayofyear
df["week_of_year"] = df["fl_date"].dt.isocalendar().week.astype(int)

# Cyclical encoding for month and day_of_week
df["month_sin"] = np.sin(2 * np.pi * df["MONTH"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["MONTH"] / 12)
df["dow_sin"]   = np.sin(2 * np.pi * df["DAY_OF_WEEK"] / 7)
df["dow_cos"]   = np.cos(2 * np.pi * df["DAY_OF_WEEK"] / 7)

/tmp/ipykernel_373111/1099356247.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["fl_date"] = pd.to_datetime(df["FL_DATE"])


## 3. Departure Time Features

In [4]:
dep_col = "CRS_DEP_TIME"  # HHMM integer

df["dep_hour"] = df[dep_col] // 100
df["dep_min"]  = df[dep_col] % 100
df["dep_minutes_since_midnight"] = df["dep_hour"] * 60 + df["dep_min"]

df["is_peak_hour"] = df["dep_hour"].between(7, 9) | df["dep_hour"].between(16, 19)
df["is_red_eye"]   = df["dep_hour"] < 6

## 4. Route & Carrier Statistics (Historical Delay Rate)

Compute delay rate from the training split only to prevent leakage.

In [5]:
TARGET = "ARR_DEL15"  # 1 = arrival delayed ≥15 min

# Chronological train/test split: train on 2023-2024, test on 2025
train_mask = df["YEAR"] <= 2024

print(f"Train rows: {train_mask.sum():,}  |  Test rows: {(~train_mask).sum():,}")

Train rows: 13,926,980  |  Test rows: 7,001,619


In [6]:
df["route"] = df["ORIGIN"] + "→" + df["DEST"]

# Use train_mask on df directly so the route column is available
route_delay_rate   = df[train_mask].groupby("route")[TARGET].mean().rename("route_delay_rate")
carrier_delay_rate = df[train_mask].groupby("OP_CARRIER")[TARGET].mean().rename("carrier_delay_rate")

df = df.join(route_delay_rate, on="route").join(carrier_delay_rate, on="OP_CARRIER")

global_mean = df[train_mask][TARGET].mean()
df["route_delay_rate"].fillna(global_mean, inplace=True)
df["carrier_delay_rate"].fillna(global_mean, inplace=True)

/tmp/ipykernel_373111/1599160921.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["route_delay_rate"].fillna(global_mean, inplace=True)
/tmp/ipykernel_373111/1599160921.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method 

0           0.148695
1           0.148695
2           0.148695
3           0.148695
4           0.148695
              ...   
20928594    0.135768
20928595    0.135768
20928596    0.135768
20928597    0.135768
20928598    0.135768
Name: carrier_delay_rate, Length: 20928599, dtype: float64

## 5. Flight Sequence / Tail Number Features

Tail-number propagation captures aircraft-level lateness cascades.

In [7]:
if "TAIL_NUM" in df.columns:
    df_sorted = df.sort_values(["TAIL_NUM", "fl_date", "CRS_DEP_TIME"])
    # DEP_DELAY_NEW is departure delay capped at 0 for early departures
    df["prev_dep_delay"] = df_sorted.groupby("TAIL_NUM")["DEP_DELAY_NEW"].shift(1)
    df["prev_dep_delay"].fillna(0, inplace=True)

/tmp/ipykernel_373111/3276562696.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["prev_dep_delay"].fillna(0, inplace=True)


## 6. Select Final Feature Set

In [8]:
FEATURE_COLS = [
    "MONTH", "month_sin", "month_cos",
    "DAY_OF_WEEK", "dow_sin", "dow_cos",
    "dep_hour", "dep_minutes_since_midnight",
    "is_peak_hour", "is_red_eye",
    "CRS_ELAPSED_TIME",
    "DISTANCE",
    "route_delay_rate",
    "carrier_delay_rate",
    # Categorical (handled natively by LightGBM, or label-encoded for XGBoost)
    "ORIGIN", "DEST", "OP_CARRIER",
]

print("Features:", len(FEATURE_COLS))
df[FEATURE_COLS + [TARGET]].head()

Features: 17


,MONTH,month_sin,month_cos,DAY_OF_WEEK,dow_sin,dow_cos,dep_hour,dep_minutes_since_midnight,is_peak_hour,is_red_eye,CRS_ELAPSED_TIME,DISTANCE,route_delay_rate,carrier_delay_rate,ORIGIN,DEST,OP_CARRIER,ARR_DEL15
0,1,0.5,0.866025,1,0.781831,0.62349,14,840,False,False,146.0,765.0,0.215374,0.148695,BNA,JFK,9E,0.0
1,1,0.5,0.866025,1,0.781831,0.62349,10,630,False,False,165.0,765.0,0.206889,0.148695,JFK,BNA,9E,0.0
2,1,0.5,0.866025,1,0.781831,0.62349,18,1139,True,False,149.0,665.0,0.168045,0.148695,JFK,IND,9E,0.0
3,1,0.5,0.866025,1,0.781831,0.62349,5,300,False,True,120.0,382.0,0.157074,0.148695,BGR,JFK,9E,0.0
4,1,0.5,0.866025,1,0.781831,0.62349,12,735,False,False,122.0,589.0,0.123461,0.148695,ATL,XNA,9E,0.0


## 7. Save Processed Dataset

In [9]:
# Drop cancelled/diverted flights — they have no ARR_DEL15 label
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)
print(f"After dropping NaN targets: {len(df):,}")

After dropping NaN targets: 20,588,154


In [10]:
out_dir = config.DATA_PART3_PROCESSED
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "flights_2023_2025_features.parquet"
df[FEATURE_COLS + [TARGET, "YEAR"]].to_parquet(out_path, index=False)
print(f"Saved: {out_path}")

Saved: /home/hareee234/Dev/sjsu/cmpe188-data/part3/processed/flights_2023_2025_features.parquet
